In [0]:
%run ./01-ATSConfigs

In [0]:
%run ./03-Endpoints

In [0]:
%run ./04-VectorSearch

In [0]:
class ProfileIndex:
    def __init__(self):
        self.vs = VectorSearch()

    def get_sync_date(self):
        sync_date = (spark.sql(
            f"""
            select date_add(last_load_date,1) as sync_date
            from {conf.jobs_metadata_table_name}
            where job_name = '{conf.index_sync_job_name}'
            order by last_load_date desc
            """
        ).first()
        .asDict()['sync_date']
        .strftime('%Y-%m-%d %H:%M:%S'))

    def update_metadata(self,sync_date):
        spark.sql(
            f""" insert into table {conf.jobs_metadata_table_name}
            values({conf.index_sync_job_name},{sync_date}
            current_timestamp(),"job execution")
            """
        )

    def index_profiles(self):
        sync_date = get_sync_date()
        self.vs.sync_index(conf.vector_index_name)
        self.update_metadata()